In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
working_directory = "/Users/kemalinecik/git_nosync/sctram"

In [3]:
import sys
sys.path.append(working_directory)

import logging
import os
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
from sctram.generate.real import sc_norman_sciplex_cpa

In [4]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.input import read_dict

2025-02-25 21:33:33.988 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /Users/kemalinecik/git_nosync/sctram/sctram/api/_defaults.yaml
2025-02-25 21:33:33.988 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [5]:
dataset_dir = "/Users/kemalinecik/git_nosync/sctram/__temp__/data"
adata_norman = sc_norman_sciplex_cpa(dataset_dir=dataset_dir)
adata_bms = adata_norman[["bms" in i.lower() or "vehicle" in i.lower() for i in adata_norman.obs["drug"]]]
adata = ad.AnnData(X=adata_bms.obsm["tardis"].copy(), obs=adata_bms.obs.copy())

2025-02-25 21:33:34.045 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/Users/kemalinecik/git_nosync/sctram/__temp__/data/norman_sciplex_cpa.h5ad') already exists. Skipping download.


In [6]:
ground_truth_trajectories = {
    "trajectory_1": [
        ('Vehicle_1.0', 'BMS_0.001'),
        ('BMS_0.001', 'BMS_0.005'),
        ('BMS_0.005', 'BMS_0.01'),
        ('BMS_0.01', 'BMS_0.05'),
        ('BMS_0.05', 'BMS_0.1'),
        ('BMS_0.1', 'BMS_0.5'),
        ('BMS_0.5', 'BMS_1.0'),
    ],
}
input_trajectories_all = read_dict(ground_truth_trajectories)
input_trajectories = input_trajectories_all.get_trajectory("trajectory_1", include_additional_nodes=False)

In [7]:
api = TrajectoryEvaluationAPI(
    adata=adata,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="DEBUG"
)

In [8]:
api.evaluate_with_defaults()

2025-02-25 21:33:34.366 | INFO     | sctram.api._lower_level:evaluate_adjacency:154 - Starting adjacency evaluation.
2025-02-25 21:33:34.366 | INFO     | sctram.api._lower_level:_get_inference_method:79 - Running pseudotime inference with method 'PAGAInference'
2025-02-25 21:33:34.367 | INFO     | sctram.api._lower_level:_get_evaluate_method:72 - Running pseudotime evaluation with method 'AdjacencyMatrixEvaluation'
2025-02-25 21:33:34.372 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:161 - Initializing from AnnData without precomputed neighbors.
2025-02-25 21:33:34.374 | DEBUG    | sctram.infer._base:_add_labels_to_adata:220 - Adding provided labels to AnnData object.
2025-02-25 21:33:34.374 | INFO     | sctram.infer._base:_initialize_from_adata_without_neighbors:168 - No precomputed neighbors found in AnnData.
2025-02-25 21:33:34.375 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:172 - AnnData initialized successfully from AnnData w

Diffusion pseudotime converged in 21 steps.


2025-02-25 21:33:39.211 | INFO     | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:47 - Metric: 'Concordance Index (CI)' (Pseudotime-based), Score: '0.08681762168273767', Computation Time: '0.2431 sec'
2025-02-25 21:33:39.211 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:37 - Calculating metric: 'dtw_distance'
2025-02-25 21:33:39.852 | INFO     | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:47 - Metric: 'Dynamic Time Warping (DTW) distance' (Pseudotime-based), Score: '0.2161888035193305', Computation Time: '0.6409 sec'
2025-02-25 21:33:39.853 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:37 - Calculating metric: 'wasserstein_distance'
2025-02-25 21:33:39.855 | INFO     | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:47 - Metric: 'Wasserstein distance' (Pseudotime-based), Score: '0.2779129512736693', Computation Time: '0.0016 sec'
2025-0

In [9]:
adata_scvi = ad.AnnData(X=adata_bms.obsm["scvi"].copy(), obs=adata_bms.obs.copy())
api_scvi = TrajectoryEvaluationAPI(
    adata=adata_scvi,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="WARNING"
)
api_scvi.evaluate_with_defaults()

2025-02-25 21:33:42.962 | WARNING  | sctram.utils._utils:sget:34 - Default value 0.5 used for missing key 'alpha'.
2025-02-25 21:33:42.962 | WARNING  | sctram.utils._utils:sget:34 - Default value 100 used for missing key 'n_steps'.
2025-02-25 21:33:42.963 | WARNING  | sctram.utils._utils:sget:34 - Default value 1e-06 used for missing key 'tol'.


Diffusion pseudotime converged in 21 steps.


In [10]:
df = api.get_all_results()
df_scvi = api_scvi.get_all_results()
df["score_tardis"] = df["score"]
df["score_scvi"] = df_scvi["score"]
del df["score"]
df

,path,metric,score_tardis,score_scvi
0,adjacency,frobenius,2.649362,3.758729
1,adjacency,l1_norm,12.775805,22.495203
2,adjacency,accuracy,0.875000,0.625000
3,adjacency,graph_edit_distance,4.000000,12.000000
4,adjacency,spectral_distance,1.793765,2.729414
5,adjacency,jaccard_similarity,0.500000,0.294118
6,adjacency,hamming_distance,8.000000,24.000000
7,adjacency,precision,0.800000,0.333333
8,adjacency,recall,0.571429,0.714286
9,adjacency,f1_score,0.666667,0.454545
